In [ ]:
# 이미지 처리와 시각화에 필요한 OpenCV 및 Matplotlib를 설치합니다.
!pip install opencv-python==4.11.0.86 matplotlib==3.9.4

In [ ]:
# Python의 모듈 검색 경로를 수정하기 위해 sys를 불러옵니다.
import sys
# RetinaFace 프로젝트 폴더를 사용자 모듈 검색 경로에 추가합니다.
sys.path.append(r'c:\ai_project01\Pytorch_Retinaface')

In [ ]:
# GPU 텐서 연산을 위해 PyTorch를 불러옵니다.
import torch
# 이미지 읽기와 색상 처리를 위해 OpenCV를 불러옵니다.
import cv2
# 배열과 좌표 계산을 위해 NumPy를 불러옵니다.
import numpy as np

# RetinaFace 모델 클래스를 불러옵니다.
from models.retinaface import RetinaFace
# ResNet-50용 RetinaFace 설정을 불러옵니다.
from data import cfg_re50
# 이미지 크기에 맞는 기본 박스를 생성하는 클래스를 불러옵니다.
from layers.functions.prior_box import PriorBox
# 겹치는 얼굴 상자를 제거하는 NMS 함수를 불러옵니다.
from utils.nms.py_cpu_nms import py_cpu_nms
# 모델의 상대 좌표를 얼굴 상자 좌표로 변환하는 함수를 불러옵니다.
from utils.box_utils import decode

# 폴더와 파일 경로를 다루기 위해 os를 불러옵니다.
import os

In [ ]:
# 현재 Python 환경에서 CUDA GPU를 사용할 수 있는지 확인합니다.
torch.cuda.is_available()

True

In [ ]:
# 모델과 입력 데이터를 실행할 장치로 CUDA GPU를 지정합니다.
device = "cuda"
# RetinaFace에 사용할 ResNet-50 설정을 선택합니다.
cfg = cfg_re50
# 테스트 단계의 RetinaFace 모델을 만들고 GPU로 이동합니다.
net = RetinaFace(cfg=cfg, phase='test').to(device)

c:\Users\user\anaconda3\envs\yolo_env01\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\user\anaconda3\envs\yolo_env01\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
# 학습된 RetinaFace 가중치 파일의 위치를 지정합니다.
pretrained_path = r'c:\ai_project01\Pytorch_Retinaface\weights\Resnet50_Final.pth'
# 가중치를 불러와 현재 실행 장치에 맞게 배치합니다.
state_dict = torch.load(pretrained_path, map_location=device)

C:\Users\user\AppData\Local\Temp\ipykernel_16532\1137602229.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_path, map_location=device)

In [ ]:
# 모델에 적용할 새로운 가중치 딕셔너리를 준비합니다.
new_state_dic = {}
# 저장된 모든 가중치 이름과 텐서를 하나씩 확인합니다.
for k, v in state_dict.items():
    # 여러 GPU 학습으로 저장된 가중치인지 확인합니다.
    if k.startswith("module."):
        # module. 접두사를 제거해 현재 모델의 이름 형식에 맞춥니다.
        new_state_dic[k[7:]] = v
    else:
        # 접두사가 없는 가중치는 이름을 그대로 저장합니다.
        new_state_dic[k] = v

In [ ]:
# 정리한 가중치를 RetinaFace 모델에 적용합니다.
net.load_state_dict(new_state_dic, strict=True)

<All keys matched successfully>

In [ ]:
# 모델을 평가 모드로 전환해 추론에 적합한 동작을 사용합니다.
net.eval()

RetinaFace(
  (body): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Seque

In [ ]:
# 클래스별로 처리할 이미지 폴더의 경로를 지정합니다.
image_dirs = {
    # 마스크를 착용한 이미지가 저장된 폴더입니다.
    'mask_on': r'c:\ai_project01\mask_images\mask_on',
    # 마스크를 착용하지 않은 이미지가 저장된 폴더입니다.
    'no_mask': r'c:\ai_project01\mask_images\no_mask',
}

In [ ]:
# YOLO 라벨 파일을 저장할 폴더 경로를 지정합니다.
label_dir = r'c:\ai_project01\labels'
# 라벨 폴더가 없으면 만들고, 이미 있으면 그대로 사용합니다.
os.makedirs(label_dir, exist_ok=True)

# 각 클래스 이름을 YOLO의 숫자 클래스 ID로 매핑합니다.
class_ids = {
    'mask_on': 0,  # 마스크 착용 클래스
    'no_mask': 1   # 마스크 미착용 클래스
}

In [ ]:
# 클래스별 이미지 폴더를 순서대로 처리합니다.
for mask_status, image_dir in image_dirs.items():
    # 현재 폴더에 있는 모든 파일을 하나씩 확인합니다.
    for img_file in os.listdir(image_dir):
        # PNG, JPG, JPEG 이미지 파일만 처리합니다.
        if img_file.lower().endswith(('png', 'jpg', 'jpeg')):
            # 현재 이미지의 전체 경로를 만듭니다.
            image_path = os.path.join(image_dir, img_file)
            # 이미지를 BGR 컬러 형식으로 읽습니다.
            img_raw = cv2.imread(image_path, cv2.IMREAD_COLOR)
            # 모델 입력 전처리를 위해 이미지 자료형을 float32로 바꿉니다.
            img = np.float32(img_raw)
            # 이미지 높이와 너비를 저장합니다.
            im_height, im_width, _ = img.shape

            # 정규화 좌표를 픽셀 좌표로 바꾸기 위한 크기 벡터를 만듭니다.
            scale = torch.Tensor([im_width, im_height, im_width, im_height]).to(device)
            # RetinaFace 학습 방식에 맞게 BGR 평균값을 뺍니다.
            img -= (104, 117, 123)
            # 이미지 차원을 높이-너비-채널에서 채널-높이-너비로 변경합니다.
            img = img.transpose(2, 0, 1)
            # 배치 차원을 추가하고 전처리한 이미지를 GPU로 이동합니다.
            img = torch.from_numpy(img).unsqueeze(0).to(device)
            # 모델로 얼굴 위치, 신뢰도, 랜드마크 예측값을 계산합니다.
            loc, conf, landms = net(img)

            # 현재 이미지 크기에 맞는 기본(anchor) 박스 생성기를 만듭니다.
            priorbox = PriorBox(cfg, image_size=(im_height, im_width))
            # 기본 박스를 생성하고 모델과 같은 장치로 이동합니다.
            priors = priorbox.forward().to(device)
            # 기본 박스 텐서의 실제 데이터를 가져옵니다.
            prior_data = priors.data

            # 상대 위치 예측값을 얼굴 상자 좌표로 복원합니다.
            boxes = decode(loc.data.squeeze(0), prior_data, cfg["variance"])
            # 정규화된 상자 좌표를 이미지 픽셀 좌표로 변환합니다.
            boxes = boxes * scale

            # 각 후보 상자에서 얼굴일 확률을 CPU NumPy 배열로 가져옵니다.
            scores = conf.squeeze(0).data.cpu().numpy()[:, 1]

            # 얼굴로 인정할 최소 신뢰도 기준을 설정합니다.
            confience_threshold = 0.5
            # 기준보다 높은 후보 상자의 인덱스를 선택합니다.
            top_indices = np.where(scores > confience_threshold)[0]

            # 신뢰도가 낮은 얼굴 상자를 제거합니다.
            boxes = boxes[top_indices]
            # 선택된 상자에 대응하는 신뢰도만 남깁니다.
            scores = scores[top_indices]
            # 상자 좌표와 신뢰도를 하나의 배열로 합칩니다.
            dets = np.hstack((boxes.cpu().numpy(), scores[:, np.newaxis])).astype(np.float32, copy=False)
            # IoU 0.3 기준으로 겹치는 중복 상자의 인덱스를 계산합니다.
            keep = py_cpu_nms(dets, 0.3)
            # NMS를 통과한 최종 탐지 결과만 남깁니다.
            dets = dets[keep, :]

            # 현재 이미지의 YOLO 라벨 문자열을 저장할 목록입니다.
            yolo_labels = []

            # 최종 얼굴 상자를 하나씩 YOLO 라벨로 변환합니다.
            for b in dets:
                # 상자의 왼쪽 위와 오른쪽 아래 픽셀 좌표를 가져옵니다.
                x1, y1, x2, y2 = b[:4]
                # YOLO 형식에 맞는 중심 x 좌표로 정규화합니다.
                x_center = ((x1 + x2) / 2) / im_width
                # YOLO 형식에 맞는 중심 y 좌표로 정규화합니다.
                y_center = ((y1 + y2) / 2) / im_height
                # 상자 너비를 이미지 너비로 나누어 정규화합니다.
                bbox_width = (x2 - x1) / im_width
                # 상자 높이를 이미지 높이로 나누어 정규화합니다.
                bbox_height = (y2 - y1) / im_height
                # 현재 이미지 폴더의 클래스 ID를 가져옵니다.
                class_id = class_ids[mask_status]
                # 클래스 ID와 정규화된 좌표를 YOLO 한 줄 형식으로 저장합니다.
                yolo_labels.append(f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}")

            # 이미지와 같은 이름의 TXT 라벨 파일 경로를 만듭니다.
            label_path = os.path.join(label_dir, os.path.splitext(img_file)[0] + '.txt')
            # 라벨 파일을 새로 쓰기 모드로 엽니다.
            with open(label_path, 'w') as f:
                # 여러 탐지 결과를 줄바꿈으로 연결해 파일에 기록합니다.
                f.write('\n'.join(yolo_labels))
            # 처리한 이미지와 저장된 라벨 파일의 경로를 출력합니다.
            print(f"Processed: {img_file} -> {label_path}")

Processed : mask_on_00000.png -> c:\ai_project01\labels\mask_on_00000.txt
Processed : mask_on_00001.png -> c:\ai_project01\labels\mask_on_00001.txt
Processed : mask_on_00002.png -> c:\ai_project01\labels\mask_on_00002.txt
Processed : mask_on_00003.png -> c:\ai_project01\labels\mask_on_00003.txt
Processed : mask_on_00004.png -> c:\ai_project01\labels\mask_on_00004.txt
Processed : mask_on_00005.png -> c:\ai_project01\labels\mask_on_00005.txt
Processed : mask_on_00006.png -> c:\ai_project01\labels\mask_on_00006.txt
Processed : mask_on_00007.png -> c:\ai_project01\labels\mask_on_00007.txt
Processed : mask_on_00008.png -> c:\ai_project01\labels\mask_on_00008.txt
Processed : mask_on_00009.png -> c:\ai_project01\labels\mask_on_00009.txt
Processed : mask_on_00010.png -> c:\ai_project01\labels\mask_on_00010.txt
Processed : mask_on_00011.png -> c:\ai_project01\labels\mask_on_00011.txt
Processed : mask_on_00012.png -> c:\ai_project01\labels\mask_on_00012.txt
Processed : mask_on_00013.png -> c:\ai